# 🎯 Ćwiczenie 5: MLlib w Spark — budowanie modeli ML na dużych zbiorach danych

**Kurs:** Python w Zastosowaniach Big Data — PJATK  
**Data:** 17.05.2026  
**Czas realizacji:** ⏱️ 90 minut  
**Poziom trudności:** ⭐⭐–⭐⭐⭐ (Intermediate-Advanced)  

---

## 📚 Cele kształcenia

Po ukończeniu tego notebooka będziesz potrafił/a:

1. **Eksplorować duże zbiory danych** z PySpark DataFrame API
2. **Przygotowywać cechy** (features) za pomocą `StringIndexer`, `OneHotEncoder` i `VectorAssembler`
3. **Budować Pipeline** przetwarzania danych w Spark MLlib
4. **Trenować modele klasyfikacji** — LogisticRegression, DecisionTree, RandomForest, GBTClassifier
5. **Ewaluować i porównywać** modele za pomocą AUC, accuracy i F1-score
6. **Optymalizować hiperparametry** z `CrossValidator` i `ParamGridBuilder`

## 📋 Wymagania wstępne

- Podstawy PySpark (DataFrames, transformacje, akcje)
- Znajomość scikit-learn (Pipeline, train/test split, metryki)
- Podstawy ML (klasyfikacja binarna, overfitting, cross-validation)

---

> 💡 **Wskazówka:** Uruchamiaj komórki po kolei od góry do dołu za pomocą `Shift+Enter`.  
> Jeśli utkniesz — kliknij komórkę z podpowiedzią (💡) lub zapytaj prowadzącego!

In [ ]:
# @title 🔧 Konfiguracja środowiska — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

# PySpark — sesja i typy
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType
)

# PySpark MLlib — przetwarzanie cech
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml import Pipeline

# PySpark MLlib — modele
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
)

# PySpark MLlib — ewaluacja i tuning
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Wizualizacja
import matplotlib.pyplot as plt
import numpy as np

# Inicjalizacja SparkSession
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('CW05-MLlib-TelekomPL')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

print(f'✅ PySpark {spark.version} gotowy!')
print(f'📌 SparkSession: {spark.sparkContext.appName}')
print('🚀 Środowisko skonfigurowane — zaczynamy!')

## 📖 Wprowadzenie — Spark MLlib vs scikit-learn

Znasz już scikit-learn — świetną bibliotekę do ML. Ale co zrobić, gdy masz **miliony** rekordów, które nie mieszczą się w pamięci jednej maszyny? 🤔

Tutaj wchodzi **Spark MLlib** — rozproszony framework do Machine Learning, który działa na klastrach setek maszyn!

### 🔑 Kluczowe pojęcia

| Pojęcie | Opis | Odpowiednik w scikit-learn |
|---------|------|---------------------------|
| **Transformer** | Przekształca DataFrame w inny DataFrame | `fit_transform()` |
| **Estimator** | Uczy się z danych i tworzy Model (Transformer) | `fit()` |
| **Pipeline** | Łańcuch Transformerów i Estimatorów | `sklearn.pipeline.Pipeline` |
| **StringIndexer** | Zamienia string → indeks liczbowy | `LabelEncoder` |
| **OneHotEncoder** | Zamienia indeks → wektor binarny (sparse) | `sklearn.preprocessing.OneHotEncoder` |
| **VectorAssembler** | Łączy wiele kolumn w jeden wektor cech | `np.column_stack()` / `ColumnTransformer` |
| **StandardScaler** | Normalizacja (mean=0, std=1) | `sklearn.preprocessing.StandardScaler` |
| **CrossValidator** | K-fold cross-validation z grid search | `GridSearchCV` |
| **ParamGridBuilder** | Siatka hiperparametrów do przeszukania | `ParameterGrid` |
| **BinaryClassificationEvaluator** | AUC-ROC dla klasyfikacji binarnej | `roc_auc_score` |

### ⚡ scikit-learn vs Spark MLlib — kiedy co wybrać?

| Cecha | scikit-learn | Spark MLlib |
|-------|-------------|-------------|
| Rozmiar danych | Do ~1M wierszy | Miliony → miliardy |
| Przetwarzanie | Jedna maszyna (RAM) | Klaster rozproszony |
| API | NumPy arrays | Spark DataFrames |
| Algorytmy | Bardzo dużo | Mniej, ale rozproszone |
| Szybkość (small data) | Szybszy | Wolniejszy (overhead) |
| Szybkość (big data) | Nie zmieści się w RAM ❌ | Skaluje horyzontalnie ✅ |

### 📱 Kontekst biznesowy

Pracujesz jako data scientist w **TelekomPL** — największym operatorze telekomunikacyjnym w Polsce 📱. Zarząd chce przewidzieć, którzy klienci odejdą do konkurencji (**customer churn**). Masz **500 000 rekordów** — za dużo na scikit-learn, czas na Spark MLlib!

**Cel:** Zbudować model klasyfikacji binarnej, który przewidzi, czy klient odejdzie (`churn = 1`) czy zostanie (`churn = 0`).

Gotowy/a? Zaczynamy! 💪

## 📊 Generowanie danych — uruchom bez modyfikacji

Generujemy syntetyczny zbiór danych klientów TelekomPL z **500 000** rekordów. Dane odzwierciedlają realistyczny scenariusz — klienci z krótkim stażem i kontraktem miesięcznym mają większą szansę na odejście. Uruchom poniższą komórkę bez zmian.

In [ ]:
# @title 📊 Generowanie danych TelekomPL — uruchom bez modyfikacji

import random

SEED = 42
N = 500_000

random.seed(SEED)
np.random.seed(SEED)

# --- Generowanie danych w NumPy (szybko) ---
ages = np.random.randint(18, 81, size=N)
genders = np.random.choice(['M', 'F'], size=N)
tenure_months = np.random.randint(1, 73, size=N)
monthly_charges = np.round(np.random.uniform(20.0, 150.0, size=N), 2)
contract_types = np.random.choice(
    ['Miesięczny', 'Roczny', 'Dwuletni'], size=N, p=[0.50, 0.30, 0.20]
)
internet_services = np.random.choice(
    ['DSL', 'Fiber', 'Brak'], size=N, p=[0.40, 0.45, 0.15]
)
phone_services = np.random.choice(['Tak', 'Nie'], size=N, p=[0.75, 0.25])
num_support_tickets = np.random.randint(0, 21, size=N)
payment_methods = np.random.choice(
    ['Przelew', 'Karta', 'E-portfel'], size=N, p=[0.40, 0.35, 0.25]
)

# total_charges = tenure * monthly (z szumem)
total_charges = np.round(
    tenure_months * monthly_charges * np.random.uniform(0.9, 1.1, size=N), 2
)

# --- Realistyczny churn (~25%) ---
# Używamy logistycznej funkcji prawdopodobieństwa
churn_score = (
    -1.5
    + 0.8 * (contract_types == 'Miesięczny').astype(float)
    - 0.5 * (contract_types == 'Dwuletni').astype(float)
    + 0.3 * (internet_services == 'Fiber').astype(float)
    - 0.02 * tenure_months
    + 0.01 * monthly_charges
    + 0.08 * num_support_tickets
    - 0.01 * ages
    + np.random.normal(0, 0.5, size=N)
)
churn_prob = 1.0 / (1.0 + np.exp(-churn_score))
churns = (np.random.random(N) < churn_prob).astype(int)

print(f'📊 Wygenerowane rekordy: {N:,}')
print(f'📈 Współczynnik churn: {churns.mean():.1%}')

# --- Konwersja do Spark DataFrame ---
# Budujemy listę wierszy partiami dla wydajności
rows = []
for i in range(N):
    rows.append((
        f'CUST-{i + 1:07d}',
        int(ages[i]),
        str(genders[i]),
        int(tenure_months[i]),
        float(monthly_charges[i]),
        float(total_charges[i]),
        str(contract_types[i]),
        str(internet_services[i]),
        str(phone_services[i]),
        int(num_support_tickets[i]),
        str(payment_methods[i]),
        int(churns[i]),
    ))

schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('age', IntegerType(), False),
    StructField('gender', StringType(), False),
    StructField('tenure_months', IntegerType(), False),
    StructField('monthly_charges', FloatType(), False),
    StructField('total_charges', FloatType(), False),
    StructField('contract_type', StringType(), False),
    StructField('internet_service', StringType(), False),
    StructField('phone_service', StringType(), False),
    StructField('num_support_tickets', IntegerType(), False),
    StructField('payment_method', StringType(), False),
    StructField('churn', IntegerType(), False),
])

df = spark.createDataFrame(rows, schema=schema)

# Cache dla szybszych kolejnych operacji
df.cache()
print(f'✅ Spark DataFrame utworzony: {df.count():,} wierszy, {len(df.columns)} kolumn')
df.show(5, truncate=False)

## 🎓 Przykład prowadzony — od surowych danych do predykcji

Zanim przejdziesz do samodzielnych ćwiczeń, prześledzimy razem **mini-pipeline** — od jednej kolumny kategorycznej do prostego modelu. To pokaże Ci pełny flow pracy w Spark MLlib.

### Cel: przewidzieć churn używając TYLKO `contract_type` i `monthly_charges`

In [ ]:
# @title 🎓 Przykład prowadzony — mini Pipeline od A do Z

# KROK 1: StringIndexer — zamienia "Miesięczny" / "Roczny" / "Dwuletni" → 0.0 / 1.0 / 2.0
indexer_demo = StringIndexer(
    inputCol='contract_type',
    outputCol='contract_type_idx'
)

# KROK 2: OneHotEncoder — zamienia indeks → wektor binarny
encoder_demo = OneHotEncoder(
    inputCol='contract_type_idx',
    outputCol='contract_type_vec'
)

# KROK 3: VectorAssembler — łączy kolumny w jeden wektor
assembler_demo = VectorAssembler(
    inputCols=['contract_type_vec', 'monthly_charges'],
    outputCol='features'
)

# KROK 4: Model — LogisticRegression
lr_demo = LogisticRegression(
    featuresCol='features',
    labelCol='churn',
    maxIter=10
)

# KROK 5: Budujemy Pipeline — automatyczny łańcuch transformacji
pipeline_demo = Pipeline(stages=[
    indexer_demo,
    encoder_demo,
    assembler_demo,
    lr_demo,
])

# Podział danych
train_demo, test_demo = df.randomSplit([0.8, 0.2], seed=42)

# Trenujemy cały Pipeline jednym wywołaniem!
model_demo = pipeline_demo.fit(train_demo)

# Predykcja
preds_demo = model_demo.transform(test_demo)
preds_demo.select('customer_id', 'contract_type', 'monthly_charges', 'churn', 'prediction', 'probability').show(10, truncate=False)

# Ewaluacja
evaluator_demo = BinaryClassificationEvaluator(
    labelCol='churn',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
auc_demo = evaluator_demo.evaluate(preds_demo)
print(f'\n📊 AUC mini-modelu (tylko 2 cechy): {auc_demo:.4f}')
print('👉 Teraz Twoja kolej — zbuduj pełny model ze wszystkimi cechami!')

---

## ✏️ Ćwiczenie 1: Eksploracja danych z PySpark ⭐

Zanim zbudujesz jakikolwiek model, musisz **poznać swoje dane**! Każdy data scientist zaczyna od eksploracji.

### Zadanie:

1. Wyświetl schemat danych (`printSchema`)
2. Policz liczbę wierszy i zapisz w zmiennej `row_count`
3. Oblicz rozkład zmiennej docelowej `churn` (ile 0 i ile 1) — zapisz jako DataFrame w `churn_dist`
4. Oblicz podstawowe statystyki opisowe dla kolumn numerycznych (`describe`)

**Oczekiwany wynik:**
- `row_count` = 500 000
- `churn_dist` — DataFrame z kolumnami `churn` i `count`

In [ ]:
# ✏️ Ćwiczenie 1: Eksploracja danych z PySpark

# TODO 1: Wyświetl schemat danych
...

# TODO 2: Policz wiersze
row_count = ...  # Zastąp ...
print(f'Liczba wierszy: {row_count:,}')

# TODO 3: Rozkład zmiennej churn — groupBy + count + orderBy
churn_dist = ...  # Zastąp ...
churn_dist.show()

# TODO 4: Statystyki opisowe dla kolumn numerycznych
numeric_cols = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets']
...

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 1
print("""
Podpowiedź 1: Schemat → df.printSchema()
Podpowiedź 2: Liczba wierszy → df.count()
Podpowiedź 3: Rozkład churn → df.groupBy('churn').count().orderBy('churn')
Podpowiedź 4: Statystyki → df.select(numeric_cols).describe().show()
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 1
try:
    assert row_count is not ..., '❌ Nie uzupełniłeś/aś row_count'
    assert row_count == 500_000, f'❌ Oczekiwano 500 000 wierszy, masz {row_count}'
    print('✅ row_count — poprawnie!')

    assert churn_dist is not ..., '❌ Nie uzupełniłeś/aś churn_dist'
    _cd = churn_dist.collect()
    assert len(_cd) == 2, f'❌ churn_dist powinien mieć 2 wiersze (0 i 1), masz {len(_cd)}'
    _total = sum(r['count'] for r in _cd)
    assert _total == 500_000, f'❌ Suma count powinna = 500 000, masz {_total}'
    print('✅ churn_dist — poprawnie!')

    print('\n🎉 Ćwiczenie 1 zaliczone! Dane poznane — lecimy dalej!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 2: Przygotowanie features — StringIndexer i OneHotEncoder ⭐⭐

Modele ML w Spark nie rozumieją stringów — potrzebują **liczb**! Musimy zakodować kolumny kategoryczne.

### Przypomnienie flow:
```
"Miesięczny" → StringIndexer → 0.0 → OneHotEncoder → (1, 0)
"Roczny"     → StringIndexer → 1.0 → OneHotEncoder → (0, 1)
"Dwuletni"   → StringIndexer → 2.0 → OneHotEncoder → (0, 0)  ← referencyjna
```

### Zadanie:

1. Utwórz **StringIndexer** dla 4 kolumn: `contract_type`, `internet_service`, `payment_method`, `gender`
2. Utwórz **OneHotEncoder** dla wyników indeksowania
3. Zapisz listy indexerów i enkoderów w zmiennych `indexers` i `encoders`

**Kolumny wyjściowe:**
- StringIndexer: dodaj suffix `_idx` (np. `contract_type_idx`)
- OneHotEncoder: dodaj suffix `_vec` (np. `contract_type_vec`)

In [ ]:
# ✏️ Ćwiczenie 2: StringIndexer i OneHotEncoder

# Lista kolumn kategorycznych do zakodowania
categorical_cols = ['contract_type', 'internet_service', 'payment_method', 'gender']

# TODO 1: Utwórz listę StringIndexerów
# Każdy StringIndexer: inputCol → kolumna oryginalna, outputCol → kolumna + '_idx'
indexers = ...  # Zastąp ... — lista StringIndexer-ów

# TODO 2: Utwórz listę OneHotEncoderów
# Każdy OneHotEncoder: inputCol → kolumna + '_idx', outputCol → kolumna + '_vec'
encoders = ...  # Zastąp ... — lista OneHotEncoder-ów

# Sprawdzenie
print(f'📋 Liczba indexerów: {len(indexers)}')
print(f'📋 Liczba enkoderów: {len(encoders)}')
for idx in indexers:
    print(f'  StringIndexer: {idx.getInputCol()} → {idx.getOutputCol()}')
for enc in encoders:
    print(f'  OneHotEncoder: {enc.getInputCol()} → {enc.getOutputCol()}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 2
print("""
Podpowiedź 1: Użyj list comprehension!

indexers = [
    StringIndexer(inputCol=col, outputCol=col + '_idx')
    for col in categorical_cols
]

Podpowiedź 2: Analogicznie dla OneHotEncoder:

encoders = [
    OneHotEncoder(inputCol=col + '_idx', outputCol=col + '_vec')
    for col in categorical_cols
]
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 2
try:
    assert indexers is not ..., '❌ Nie uzupełniłeś/aś indexers'
    assert encoders is not ..., '❌ Nie uzupełniłeś/aś encoders'
    assert len(indexers) == 4, f'❌ Oczekiwano 4 indexerów, masz {len(indexers)}'
    assert len(encoders) == 4, f'❌ Oczekiwano 4 enkoderów, masz {len(encoders)}'

    _idx_inputs = sorted([idx.getInputCol() for idx in indexers])
    _expected = sorted(categorical_cols)
    assert _idx_inputs == _expected, f'❌ Indexery powinny mieć inputCol: {categorical_cols}'

    for idx in indexers:
        assert idx.getOutputCol().endswith('_idx'), f'❌ OutputCol powinien kończyć się na _idx: {idx.getOutputCol()}'
    for enc in encoders:
        assert enc.getOutputCol().endswith('_vec'), f'❌ OutputCol powinien kończyć się na _vec: {enc.getOutputCol()}'

    print('✅ Indexery i enkodery — poprawne!')
    print('\n🎉 Ćwiczenie 2 zaliczone! Cechy gotowe do kodowania!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 3: VectorAssembler i Pipeline ⭐⭐

Mamy indexery i enkodery — teraz musimy **połączyć wszystkie cechy w jeden wektor**. W Spark MLlib każdy model oczekuje jednej kolumny `features` — wektora ze wszystkimi cechami.

### Zadanie:

1. Utwórz `VectorAssembler`, który łączy:
   - Kolumny zakodowane (OHE): `contract_type_vec`, `internet_service_vec`, `payment_method_vec`, `gender_vec`
   - Kolumny numeryczne: `age`, `tenure_months`, `monthly_charges`, `total_charges`, `num_support_tickets`
   - Kolumna binarna: `phone_service_idx` (Tak=0, Nie=1 — wystarczy indexer, bez OHE)
2. Utwórz `StandardScaler` na wektorze cech
3. Zbuduj pełny preprocessing `Pipeline` ze **wszystkich** etapów:
   - `indexers` + `encoders` + `assembler` + `scaler`
4. Wytrenuj (`.fit()`) Pipeline na danych `df` i zapisz w `preprocessing_model`
5. Przetransformuj dane i zapisz w `df_prepared`

> ⚠️ **Uwaga:** `phone_service` ma tylko 2 wartości (Tak/Nie), więc wystarczy `StringIndexer` (0/1). Nie potrzebuje OneHotEncoder!

In [ ]:
# ✏️ Ćwiczenie 3: VectorAssembler i Pipeline

# Dodatkowy indexer dla phone_service (binarna kolumna — wystarczy sam indeks)
phone_indexer = StringIndexer(inputCol='phone_service', outputCol='phone_service_idx')

# TODO 1: Definiuj listę kolumn wejściowych dla VectorAssembler
# Kolumny OHE (4 sztuki końcówka '_vec') + numeryczne + phone_service_idx
numeric_features = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets']
ohe_features = [col + '_vec' for col in categorical_cols]
assembler_inputs = ...  # Zastąp ... — połącz: ohe_features + numeric_features + ['phone_service_idx']

# TODO 2: Utwórz VectorAssembler
assembler = ...  # Zastąp ... — VectorAssembler(inputCols=..., outputCol='features_raw')

# TODO 3: Utwórz StandardScaler
scaler = ...  # Zastąp ... — StandardScaler(inputCol='features_raw', outputCol='features')

# TODO 4: Zbuduj Pipeline ze wszystkimi etapami
# Kolejność: indexers + encoders + [phone_indexer] + [assembler] + [scaler]
preprocessing_pipeline = ...  # Zastąp ... — Pipeline(stages=[...])

# TODO 5: Wytrenuj Pipeline i przetransformuj dane
preprocessing_model = ...  # Zastąp ... — preprocessing_pipeline.fit(df)
df_prepared = ...  # Zastąp ... — preprocessing_model.transform(df)

# Sprawdzenie
print(f'📋 Kolumny po transformacji: {len(df_prepared.columns)}')
print(f'📊 Rozmiar wektora features:')
df_prepared.select('customer_id', 'features', 'churn').show(5, truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 3
print("""
Podpowiedź 1: Połączenie list
  assembler_inputs = ohe_features + numeric_features + ['phone_service_idx']

Podpowiedź 2: VectorAssembler
  assembler = VectorAssembler(inputCols=assembler_inputs, outputCol='features_raw')

Podpowiedź 3: StandardScaler
  scaler = StandardScaler(inputCol='features_raw', outputCol='features', withStd=True, withMean=False)
  # withMean=False, bo sparse vectors nie obsługują mean-centering

Podpowiedź 4: Pipeline — łączysz WSZYSTKIE etapy w jedną listę
  stages = indexers + encoders + [phone_indexer, assembler, scaler]
  preprocessing_pipeline = Pipeline(stages=stages)

Podpowiedź 5: Fit + Transform
  preprocessing_model = preprocessing_pipeline.fit(df)
  df_prepared = preprocessing_model.transform(df)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 3
try:
    assert assembler_inputs is not ..., '❌ Nie uzupełniłeś/aś assembler_inputs'
    assert assembler is not ..., '❌ Nie uzupełniłeś/aś assembler'
    assert scaler is not ..., '❌ Nie uzupełniłeś/aś scaler'
    assert preprocessing_pipeline is not ..., '❌ Nie uzupełniłeś/aś preprocessing_pipeline'
    assert preprocessing_model is not ..., '❌ Nie uzupełniłeś/aś preprocessing_model'
    assert df_prepared is not ..., '❌ Nie uzupełniłeś/aś df_prepared'

    # Sprawdź czy 'features' istnieje
    assert 'features' in df_prepared.columns, '❌ Brak kolumny "features" w df_prepared'
    assert 'churn' in df_prepared.columns, '❌ Brak kolumny "churn" w df_prepared'

    # Sprawdź liczbę stages
    _stages = preprocessing_pipeline.getStages()
    assert len(_stages) >= 10, f'❌ Pipeline powinien mieć ≥10 etapów (4 idx + 4 ohe + phone_idx + assembler + scaler), masz {len(_stages)}'

    # Sprawdź liczbę wierszy
    _cnt = df_prepared.count()
    assert _cnt == 500_000, f'❌ df_prepared powinien mieć 500 000 wierszy, ma {_cnt}'

    print('✅ Pipeline i VectorAssembler — poprawne!')
    print('\n🎉 Ćwiczenie 3 zaliczone! Dane przygotowane do modelowania!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 4: Podział danych i pierwszy model ⭐⭐

Mamy przygotowane dane — czas na **pierwszy model**! Zaczniemy od klasyka — **Logistic Regression**.

### Zadanie:

1. Podziel `df_prepared` na zbiór treningowy (80%) i testowy (20%) za pomocą `randomSplit` — użyj `seed=42`
2. Zapisz w zmiennych `train_data` i `test_data`
3. Wyświetl liczbę wierszy w obu zbiorach
4. Utwórz model `LogisticRegression` z parametrami:
   - `featuresCol='features'`
   - `labelCol='churn'`
   - `maxIter=20`
5. Wytrenuj model na `train_data` → zapisz w `lr_model`
6. Zrób predykcję na `test_data` → zapisz w `lr_predictions`
7. Wyświetl 10 przykładowych predykcji

In [ ]:
# ✏️ Ćwiczenie 4: Podział danych i LogisticRegression

# TODO 1: Podział danych 80/20
train_data, test_data = ...  # Zastąp ... — df_prepared.randomSplit([...], seed=42)

print(f'📊 Zbiór treningowy: {train_data.count():,} wierszy')
print(f'📊 Zbiór testowy:    {test_data.count():,} wierszy')

# TODO 2: Utwórz LogisticRegression
lr = ...  # Zastąp ...

# TODO 3: Wytrenuj model
lr_model = ...  # Zastąp ... — lr.fit(train_data)

# TODO 4: Predykcja na zbiorze testowym
lr_predictions = ...  # Zastąp ... — lr_model.transform(test_data)

# Wyświetl wyniki
lr_predictions.select(
    'customer_id', 'churn', 'prediction', 'probability'
).show(10, truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 4
print("""
Podpowiedź 1: Podział danych
  train_data, test_data = df_prepared.randomSplit([0.8, 0.2], seed=42)

Podpowiedź 2: Model
  lr = LogisticRegression(featuresCol='features', labelCol='churn', maxIter=20)

Podpowiedź 3: Trening i predykcja
  lr_model = lr.fit(train_data)
  lr_predictions = lr_model.transform(test_data)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 4
try:
    assert train_data is not ..., '❌ Nie uzupełniłeś/aś train_data'
    assert test_data is not ..., '❌ Nie uzupełniłeś/aś test_data'
    assert lr_model is not ..., '❌ Nie uzupełniłeś/aś lr_model'
    assert lr_predictions is not ..., '❌ Nie uzupełniłeś/aś lr_predictions'

    _train_cnt = train_data.count()
    _test_cnt = test_data.count()
    assert _train_cnt + _test_cnt == 500_000, f'❌ Łączna liczba powinna = 500 000'
    assert abs(_train_cnt / 500_000 - 0.8) < 0.02, f'❌ Zbiór treningowy powinien mieć ~80%, ma {_train_cnt / 500_000:.1%}'
    print(f'✅ Podział danych: train={_train_cnt:,}, test={_test_cnt:,}')

    assert 'prediction' in lr_predictions.columns, '❌ Brak kolumny prediction — czy wytrenowałeś model?'
    assert 'probability' in lr_predictions.columns, '❌ Brak kolumny probability'
    print('✅ Model LogisticRegression — wytrenowany i predykcje gotowe!')

    print('\n🎉 Ćwiczenie 4 zaliczone! Masz pierwszy model — brawo! 🏆')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 5: Ewaluacja modelu ⭐⭐

Model działa — ale **jak dobrze**? 🤔 Czas to zmierzyć!

W klasyfikacji binarnej mamy kilka kluczowych metryk:

| Metryka | Co mierzy | Kiedy ważna |
|---------|-----------|-------------|
| **AUC-ROC** | Zdolność rozróżniania klas (0.5 = losowy, 1.0 = idealny) | Ogólna jakość modelu |
| **Accuracy** | % poprawnych predykcji | Gdy klasy są zbalansowane |
| **Precision** | Z tych co model oznaczył jako churn, ile naprawdę odeszło? | Gdy koszt fałszywego alarmu wysoki |
| **Recall** | Z tych co naprawdę odeszli, ilu model złapał? | Gdy nie chcemy przegapić churna |
| **F1** | Średnia harmoniczna precision i recall | Kompromis |

### Zadanie:

1. Utwórz `BinaryClassificationEvaluator` i oblicz **AUC** — zapisz w `lr_auc`
2. Utwórz `MulticlassClassificationEvaluator` i oblicz **accuracy** → `lr_accuracy`
3. Oblicz **F1** → `lr_f1`
4. Oblicz **macierz pomyłek** (confusion matrix) — ile TP, FP, TN, FN?

In [ ]:
# ✏️ Ćwiczenie 5: Ewaluacja modelu

# TODO 1: BinaryClassificationEvaluator → AUC
binary_evaluator = BinaryClassificationEvaluator(
    labelCol='churn',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
lr_auc = ...  # Zastąp ... — binary_evaluator.evaluate(lr_predictions)
print(f'📊 AUC:      {lr_auc:.4f}')

# TODO 2: MulticlassClassificationEvaluator → accuracy
multi_evaluator_acc = ...  # Zastąp ... — MulticlassClassificationEvaluator(labelCol=..., metricName='accuracy')
lr_accuracy = ...  # Zastąp ... — multi_evaluator_acc.evaluate(lr_predictions)
print(f'📊 Accuracy: {lr_accuracy:.4f}')

# TODO 3: MulticlassClassificationEvaluator → F1
multi_evaluator_f1 = ...  # Zastąp ... — MulticlassClassificationEvaluator z metricName='f1'
lr_f1 = ...  # Zastąp ... — multi_evaluator_f1.evaluate(lr_predictions)
print(f'📊 F1-score: {lr_f1:.4f}')

# TODO 4: Macierz pomyłek (confusion matrix)
# Pogrupuj lr_predictions po (churn, prediction) i policz
confusion = ...  # Zastąp ... — lr_predictions.groupBy('churn', 'prediction').count().orderBy('churn', 'prediction')
print('\n📋 Macierz pomyłek:')
confusion.show()

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 5
print("""
Podpowiedź 1: AUC
  lr_auc = binary_evaluator.evaluate(lr_predictions)

Podpowiedź 2: Accuracy
  multi_evaluator_acc = MulticlassClassificationEvaluator(
      labelCol='churn', predictionCol='prediction', metricName='accuracy'
  )
  lr_accuracy = multi_evaluator_acc.evaluate(lr_predictions)

Podpowiedź 3: F1
  multi_evaluator_f1 = MulticlassClassificationEvaluator(
      labelCol='churn', predictionCol='prediction', metricName='f1'
  )
  lr_f1 = multi_evaluator_f1.evaluate(lr_predictions)

Podpowiedź 4: Confusion matrix
  confusion = lr_predictions.groupBy('churn', 'prediction').count().orderBy('churn', 'prediction')
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 5
try:
    assert lr_auc is not ..., '❌ Nie uzupełniłeś/aś lr_auc'
    assert lr_accuracy is not ..., '❌ Nie uzupełniłeś/aś lr_accuracy'
    assert lr_f1 is not ..., '❌ Nie uzupełniłeś/aś lr_f1'
    assert confusion is not ..., '❌ Nie uzupełniłeś/aś confusion'

    assert 0.5 <= lr_auc <= 1.0, f'❌ AUC powinno być w [0.5, 1.0], masz {lr_auc:.4f}'
    assert 0.0 <= lr_accuracy <= 1.0, f'❌ Accuracy powinno być w [0, 1], masz {lr_accuracy:.4f}'
    assert 0.0 <= lr_f1 <= 1.0, f'❌ F1 powinno być w [0, 1], masz {lr_f1:.4f}'
    print(f'✅ AUC = {lr_auc:.4f}')
    print(f'✅ Accuracy = {lr_accuracy:.4f}')
    print(f'✅ F1 = {lr_f1:.4f}')

    _conf_cnt = confusion.count()
    assert _conf_cnt == 4, f'❌ Macierz pomyłek powinna mieć 4 wiersze (TP, FP, TN, FN), ma {_conf_cnt}'
    print('✅ Macierz pomyłek — poprawna!')

    print('\n🎉 Ćwiczenie 5 zaliczone! Umiesz ewaluować modele w Spark!')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 6: Porównanie modeli ⭐⭐⭐

Logistic Regression to dopiero początek! Spark MLlib oferuje wiele algorytmów. Czas porównać **4 modele**:

| Model | Typ | Zalety |
|-------|-----|--------|
| **LogisticRegression** | Liniowy | Szybki, interpretowalny |
| **DecisionTreeClassifier** | Drzewiasty | Łatwy do interpretacji, nieliniowy |
| **RandomForestClassifier** | Ensemble | Dokładniejszy, odporny na overfitting |
| **GBTClassifier** | Gradient Boosting | Często najdokładniejszy |

### Zadanie:

1. Utwórz 3 nowe modele: `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`
2. Wytrenuj każdy na `train_data` i zrób predykcje na `test_data`
3. Oblicz AUC, accuracy i F1 dla każdego modelu
4. Złóż wyniki w tabelę porównawczą `results_df` (pandas DataFrame)

> 💡 **Tip:** GBTClassifier nie obsługuje `rawPrediction` domyślnie — użyj `rawPredictionCol='rawPrediction'` lub zmień `metricName` na `'areaUnderPR'` jeśli AUC nie zadziała. W nowszych wersjach Spark działa OK.

In [ ]:
# ✏️ Ćwiczenie 6: Porównanie modeli

import pandas as pd

# Słownik modeli do porównania
# LogisticRegression już mamy (lr) — teraz 3 kolejne

# TODO 1: Utwórz DecisionTreeClassifier
dt = ...  # Zastąp ... — DecisionTreeClassifier(featuresCol='features', labelCol='churn', maxDepth=10)

# TODO 2: Utwórz RandomForestClassifier
rf = ...  # Zastąp ... — RandomForestClassifier(featuresCol='features', labelCol='churn', numTrees=50, maxDepth=10)

# TODO 3: Utwórz GBTClassifier
gbt = ...  # Zastąp ... — GBTClassifier(featuresCol='features', labelCol='churn', maxIter=50, maxDepth=5)

# Słownik: nazwa → (model, predykcje)
models = {
    'LogisticRegression': lr,
    'DecisionTree': dt,
    'RandomForest': rf,
    'GBTClassifier': gbt,
}

# TODO 4: Wytrenuj każdy model, zrób predykcje, oblicz metryki
results = []
predictions_dict = {}  # Zapisujemy predykcje do późniejszego użycia

for name, model in models.items():
    print(f'\n🔄 Trenuję {name}...')

    # TODO: Wytrenuj model na train_data
    fitted_model = ...  # Zastąp ... — model.fit(train_data)

    # TODO: Zrób predykcje na test_data
    preds = ...  # Zastąp ... — fitted_model.transform(test_data)
    predictions_dict[name] = preds

    # TODO: Oblicz AUC, accuracy, F1
    model_auc = ...  # Zastąp ... — binary_evaluator.evaluate(preds)
    model_acc = ...  # Zastąp ... — multi_evaluator_acc.evaluate(preds)
    model_f1 = ...   # Zastąp ... — multi_evaluator_f1.evaluate(preds)

    results.append({
        'Model': name,
        'AUC': round(model_auc, 4),
        'Accuracy': round(model_acc, 4),
        'F1': round(model_f1, 4),
    })
    print(f'  ✅ AUC={model_auc:.4f}  Acc={model_acc:.4f}  F1={model_f1:.4f}')

# TODO 5: Utwórz tabelę wyników (pandas DataFrame), posortuj po AUC malejąco
results_df = ...  # Zastąp ... — pd.DataFrame(results).sort_values('AUC', ascending=False)

print('\n📊 Tabela porównawcza modeli:')
print('=' * 55)
print(results_df.to_string(index=False))

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 6
print("""
Podpowiedź 1: Modele
  dt = DecisionTreeClassifier(featuresCol='features', labelCol='churn', maxDepth=10)
  rf = RandomForestClassifier(featuresCol='features', labelCol='churn', numTrees=50, maxDepth=10)
  gbt = GBTClassifier(featuresCol='features', labelCol='churn', maxIter=50, maxDepth=5)

Podpowiedź 2: W pętli
  fitted_model = model.fit(train_data)
  preds = fitted_model.transform(test_data)
  model_auc = binary_evaluator.evaluate(preds)
  model_acc = multi_evaluator_acc.evaluate(preds)
  model_f1 = multi_evaluator_f1.evaluate(preds)

Podpowiedź 3: Tabela
  results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 6
try:
    assert dt is not ..., '❌ Nie uzupełniłeś/aś dt'
    assert rf is not ..., '❌ Nie uzupełniłeś/aś rf'
    assert gbt is not ..., '❌ Nie uzupełniłeś/aś gbt'
    assert results_df is not ..., '❌ Nie uzupełniłeś/aś results_df'

    assert isinstance(results_df, pd.DataFrame), '❌ results_df powinien być pandas DataFrame'
    assert len(results_df) == 4, f'❌ results_df powinien mieć 4 wiersze (4 modele), ma {len(results_df)}'
    assert set(results_df.columns) >= {'Model', 'AUC', 'Accuracy', 'F1'}, '❌ Brakuje kolumn: Model, AUC, Accuracy, F1'

    _models_in_df = set(results_df['Model'])
    assert 'RandomForest' in _models_in_df, '❌ Brak RandomForest w wynikach'
    assert 'GBTClassifier' in _models_in_df, '❌ Brak GBTClassifier w wynikach'

    # Sprawdź czy AUC w sensownym zakresie
    for _, row in results_df.iterrows():
        assert 0.5 <= row['AUC'] <= 1.0, f'❌ AUC dla {row["Model"]} poza zakresem: {row["AUC"]}'

    print('✅ Wszystkie 4 modele wytrenowane i ocenione!')
    print(f'🏆 Najlepszy model wg AUC: {results_df.iloc[0]["Model"]} (AUC={results_df.iloc[0]["AUC"]})')
    print('\n🎉 Ćwiczenie 6 zaliczone! Porównanko zrobione! 🔥')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## ✏️ Ćwiczenie 7: Hyperparameter Tuning z CrossValidator ⭐⭐⭐

Domyślne parametry modeli rzadko są optymalne. Czas na **tuning hiperparametrów**!

W scikit-learn używałeś `GridSearchCV`. W Spark MLlib mamy analogiczne narzędzia:

| scikit-learn | Spark MLlib |
|-------------|-------------|
| `ParameterGrid` | `ParamGridBuilder` |
| `GridSearchCV` | `CrossValidator` |
| `cv=5` | `numFolds=3` (mniej foldów, bo danych jest dużo!) |

### Zadanie:

1. Utwórz NOWY `RandomForestClassifier` — `rf_tuned`
2. Zbuduj siatkę parametrów (`ParamGridBuilder`):
   - `numTrees`: [50, 100]
   - `maxDepth`: [5, 10, 15]
3. Utwórz `CrossValidator` z 3 foldami i `BinaryClassificationEvaluator` (AUC)
4. Uruchom tuning na `train_data` — zapisz w `cv_model`
5. Pobierz najlepszy model → `best_rf_model`
6. Zrób predykcje i oblicz AUC najlepszego modelu → `best_auc`

> ⏱️ **Uwaga:** CrossValidator z 6 kombinacjami × 3 foldy = 18 trenowań. Na Colab może to potrwać ~2-5 minut.

In [ ]:
# ✏️ Ćwiczenie 7: CrossValidator + ParamGridBuilder

# TODO 1: Nowy RandomForestClassifier
rf_tuned = RandomForestClassifier(
    featuresCol='features',
    labelCol='churn',
    seed=42
)

# TODO 2: Siatka hiperparametrów
param_grid = ...  # Zastąp ... — (ParamGridBuilder()
#   .addGrid(rf_tuned.numTrees, [50, 100])
#   .addGrid(rf_tuned.maxDepth, [5, 10, 15])
#   .build())

# TODO 3: CrossValidator
crossval = ...  # Zastąp ... — CrossValidator(
#   estimator=rf_tuned,
#   estimatorParamMaps=param_grid,
#   evaluator=binary_evaluator,  # BinaryClassificationEvaluator z ćw. 5
#   numFolds=3,
#   seed=42)

# TODO 4: Uruchom tuning
print('🔄 Rozpoczynam CrossValidation — to może potrwać kilka minut...')
cv_model = ...  # Zastąp ... — crossval.fit(train_data)
print('✅ Tuning zakończony!')

# TODO 5: Najlepszy model
best_rf_model = ...  # Zastąp ... — cv_model.bestModel

# Wyświetl parametry najlepszego modelu
print(f'\n🏆 Najlepszy RandomForest:')
print(f'   numTrees = {best_rf_model.getNumTrees}')
print(f'   maxDepth = {best_rf_model.getOrDefault("maxDepth")}')

# TODO 6: Predykcje i AUC najlepszego modelu
best_predictions = ...  # Zastąp ... — best_rf_model.transform(test_data)
best_auc = ...  # Zastąp ... — binary_evaluator.evaluate(best_predictions)
best_acc = multi_evaluator_acc.evaluate(best_predictions)
best_f1 = multi_evaluator_f1.evaluate(best_predictions)

print(f'\n📊 Wyniki najlepszego modelu:')
print(f'   AUC      = {best_auc:.4f}')
print(f'   Accuracy = {best_acc:.4f}')
print(f'   F1       = {best_f1:.4f}')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 7
print("""
Podpowiedź 1: ParamGridBuilder
  param_grid = (ParamGridBuilder()
      .addGrid(rf_tuned.numTrees, [50, 100])
      .addGrid(rf_tuned.maxDepth, [5, 10, 15])
      .build())

Podpowiedź 2: CrossValidator
  crossval = CrossValidator(
      estimator=rf_tuned,
      estimatorParamMaps=param_grid,
      evaluator=binary_evaluator,
      numFolds=3,
      seed=42
  )

Podpowiedź 3: Trening
  cv_model = crossval.fit(train_data)

Podpowiedź 4: Najlepszy model
  best_rf_model = cv_model.bestModel
  best_predictions = best_rf_model.transform(test_data)
  best_auc = binary_evaluator.evaluate(best_predictions)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 7
try:
    assert param_grid is not ..., '❌ Nie uzupełniłeś/aś param_grid'
    assert crossval is not ..., '❌ Nie uzupełniłeś/aś crossval'
    assert cv_model is not ..., '❌ Nie uzupełniłeś/aś cv_model'
    assert best_rf_model is not ..., '❌ Nie uzupełniłeś/aś best_rf_model'
    assert best_auc is not ..., '❌ Nie uzupełniłeś/aś best_auc'

    # Sprawdź siatkę parametrów
    assert isinstance(param_grid, list), '❌ param_grid powinien być listą (wynik .build())'
    assert len(param_grid) == 6, f'❌ Siatka powinna mieć 2×3=6 kombinacji, ma {len(param_grid)}'
    print(f'✅ Siatka parametrów: {len(param_grid)} kombinacji')

    # Sprawdź CrossValidator
    assert crossval.getNumFolds() == 3, f'❌ Oczekiwano 3 foldów, masz {crossval.getNumFolds()}'
    print('✅ CrossValidator: 3 foldy')

    # Sprawdź AUC
    assert 0.5 <= best_auc <= 1.0, f'❌ AUC poza zakresem: {best_auc}'
    print(f'✅ Najlepszy AUC: {best_auc:.4f}')

    print('\n🎉 Ćwiczenie 7 zaliczone! Jesteś mistrzem tuningu! 🏆🔥')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie — dasz radę! 💪')
except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź swój kod i spróbuj ponownie')

---

## 🏆 Bonus: Feature Importance i wizualizacja ⭐⭐⭐

Zarząd TelekomPL nie chce tylko predykcji — chce wiedzieć **DLACZEGO** klienci odchodzą! 📱

RandomForest pozwala wyciągnąć **feature importance** — które cechy mają największy wpływ na churn.

### Zadanie:

1. Wyciągnij `featureImportances` z najlepszego modelu RandomForest
2. Dopasuj nazwy cech do indeksów wektora features
3. Utwórz pandas DataFrame z cechami i ich ważnościami
4. Zwizualizuj **top 10** najważniejszych cech (wykres słupkowy, matplotlib)
5. **Bonus w bonusie:** Zapisz najlepszy model na dysk i załaduj go z powrotem

In [ ]:
# 🏆 Bonus: Feature Importance

# TODO 1: Wyciągnij feature importance z najlepszego modelu
importances = ...  # Zastąp ... — best_rf_model.featureImportances

# TODO 2: Nazwy cech — musimy odtworzyć kolejność z VectorAssembler
# Pamiętaj: OHE tworzy n-1 kolumn per kategorię
# Prościej: użyj assembler_inputs jako nazw (nie idealne, ale wystarczające)
feature_names = assembler_inputs

# Rozkład dla OHE — jeden wektor to wiele cech
# Tworzymy uproszczoną listę nazw
# Dla wektorów OHE rozwijamy nazwy
expanded_names = []
for col_name in assembler_inputs:
    if col_name.endswith('_vec'):
        # Sprawdź ile wartości ma ten wektor
        base = col_name.replace('_vec', '')
        n_vals = df.select(base).distinct().count() - 1  # OHE daje n-1 kolumn
        for i in range(n_vals):
            expanded_names.append(f'{base}_{i}')
    else:
        expanded_names.append(col_name)

# TODO 3: DataFrame z cechami i ważnościami
importance_df = pd.DataFrame({
    'feature': expanded_names[:len(importances)],
    'importance': importances.toArray()[:len(expanded_names)]
}).sort_values('importance', ascending=False)

print('📊 Top 10 najważniejszych cech:')
print(importance_df.head(10).to_string(index=False))

# TODO 4: Wykres słupkowy top 10 cech
top10 = importance_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    top10['feature'][::-1],
    top10['importance'][::-1],
    color='#2196F3',
    edgecolor='#1565C0'
)
plt.xlabel('Ważność cechy (Feature Importance)')
plt.title('🏆 Top 10 najważniejszych cech — model RandomForest\nTelekomPL Customer Churn')
plt.tight_layout()
plt.show()

In [ ]:
# 🏆 Bonus w bonusie: Zapis i odczyt modelu

import tempfile
import os

# Zapisz model do katalogu tymczasowego
model_path = os.path.join(tempfile.gettempdir(), 'telekompl_best_rf_model')

# Zapis
best_rf_model.write().overwrite().save(model_path)
print(f'💾 Model zapisany w: {model_path}')

# Odczyt
from pyspark.ml.classification import RandomForestClassificationModel
loaded_model = RandomForestClassificationModel.load(model_path)
print(f'📂 Model załadowany! numTrees={loaded_model.getNumTrees}')

# Weryfikacja — czy załadowany model daje te same wyniki?
loaded_preds = loaded_model.transform(test_data)
loaded_auc = binary_evaluator.evaluate(loaded_preds)
print(f'📊 AUC załadowanego modelu: {loaded_auc:.4f}')
print(f'📊 AUC oryginalnego modelu: {best_auc:.4f}')
print(f'✅ Modele identyczne: {abs(loaded_auc - best_auc) < 0.0001}')

---

## 📝 Podsumowanie

Świetna robota! 🎉 Oto czego się dziś nauczyłeś/aś:

| Temat | Czego się nauczyłeś |
|-------|--------------------|
| 📊 **Eksploracja** | `printSchema()`, `describe()`, `groupBy().count()` w PySpark |
| 🔧 **Feature Engineering** | `StringIndexer` → `OneHotEncoder` → `VectorAssembler` |
| 🔗 **Pipeline** | Łączenie kroków przetwarzania w jeden obiekt `Pipeline` |
| 🤖 **Modele** | LogisticRegression, DecisionTree, RandomForest, GBT |
| 📈 **Ewaluacja** | AUC-ROC, Accuracy, F1, macierz pomyłek |
| 🎯 **Tuning** | `ParamGridBuilder` + `CrossValidator` = najlepsze hiperparametry |
| 🏆 **Feature Importance** | Interpretacja modelu — które cechy są najważniejsze |

### 🔑 Kluczowe wnioski

1. **Spark MLlib** to potężne narzędzie do ML na dużych danych — API jest bardzo podobne do scikit-learn!
2. **Pipeline** to must-have — zapewnia powtarzalność i czytelność kodu
3. **CrossValidator** automatycznie szuka najlepszych hiperparametrów
4. **Ensemble methods** (RandomForest, GBT) zazwyczaj wygrywają z prostszymi modelami
5. **Feature importance** pomaga zrozumieć, co naprawdę wpływa na wynik

### 📚 Dalsze zasoby

- [Spark MLlib Documentation](https://spark.apache.org/docs/latest/ml-guide.html)
- [PySpark ML API Reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html)
- [Machine Learning with PySpark — O'Reilly](https://www.oreilly.com/library/view/machine-learning-with/9781785889936/)

### 🧹 Sprzątanie

In [ ]:
# Zatrzymaj SparkSession
spark.stop()
print('🧹 SparkSession zatrzymana. Dobra robota! 👏')